# Rain / Snow / Hail Image Classifier

This notebook trains a convolutional neural network (transfer learning on **ResNet18**) to classify a photo as **rain**, **snow**, or **hail**.

It is built to run top-to-bottom in **Google Colab** (Runtime → Change runtime type → GPU is recommended but not required).

**Pipeline**
1. Download the [Weather Image Recognition dataset](https://www.kaggle.com/datasets/jehanbhathena/weather-dataset) from Kaggle (11 weather-phenomenon classes, derived from the WEAPD academic dataset — Xiao et al., 2021, *Earth and Space Science*).
2. Keep only the `rain`, `snow`, and `hail` classes and split them into train / validation / test sets.
3. Fine-tune a pretrained ResNet18 in two phases (frozen backbone → full fine-tune).
4. Evaluate on a held-out test set (accuracy, confusion matrix, per-class precision/recall).
5. Save the trained model to a `.pth` file (and a portable TorchScript `.pt` file) that you can download and reuse, plus a small "upload a photo and classify it" demo cell.

---
## 0. One-time setup: get a free Kaggle API key
1. Go to https://www.kaggle.com/settings/account (make a free account if you don't have one).
2. Under **API**, click **Create New Token** — this downloads a `kaggle.json` file.
3. Keep that file handy; the next cell will ask you to upload it.


In [ ]:
# 1. Install/import packages
# Colab already ships with torch/torchvision/scikit-learn/matplotlib, this just
# makes sure everything needed is present and pins the Kaggle CLI.
import sys, subprocess
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install("kaggle", "scikit-learn", "seaborn")

import os, shutil, random, json, time, copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


In [ ]:
# 2. Kaggle API key + dataset download
# Uploads kaggle.json (from the setup step above) and downloads the dataset.
IN_COLAB = "google.colab" in sys.modules

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    if IN_COLAB:
        from google.colab import files
        print("Please select the kaggle.json file you downloaded from Kaggle.")
        uploaded = files.upload()
        fname = next(iter(uploaded))
        shutil.move(fname, kaggle_json)
    else:
        raise FileNotFoundError(
            f"Put your kaggle.json at {kaggle_json} (chmod 600) before running this cell, "
            "or run this notebook in Colab."
        )
os.chmod(kaggle_json, 0o600)

DATA_ROOT = Path("data")
RAW_DIR = DATA_ROOT / "weather_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not any(RAW_DIR.iterdir()):
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "jehanbhathena/weather-dataset",
         "-p", str(RAW_DIR), "--unzip"],
        check=True,
    )
else:
    print("Dataset already downloaded, skipping.")

print("Top-level contents of", RAW_DIR, ":", os.listdir(RAW_DIR))


In [ ]:
# 3. Locate the class folders and inspect counts
# The dataset ships as one folder per class (names may be nested one level,
# e.g. data/weather_raw/dataset/rain/*.jpg) — this finds them automatically.
def find_class_dirs(root: Path):
    '''Return {class_name: Path} for every leaf directory that contains images.'''
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    class_dirs = {}
    for p in root.rglob("*"):
        if p.is_dir():
            imgs = [f for f in p.iterdir() if f.suffix.lower() in exts]
            if imgs:
                class_dirs[p.name.lower()] = p
    return class_dirs

all_class_dirs = find_class_dirs(RAW_DIR)
print("Found classes:", sorted(all_class_dirs.keys()))
for name, path in sorted(all_class_dirs.items()):
    n = len([f for f in path.iterdir() if f.is_file()])
    print(f"  {name:12s} {n:5d} images   ({path})")


In [ ]:
# 4. Keep only rain / snow / hail and split into train/val/test
TARGET_CLASSES = ["rain", "snow", "hail"]  # edit this list to add more phenomena later
SPLIT_RATIOS = (0.70, 0.15, 0.15)          # train, val, test

missing = [c for c in TARGET_CLASSES if c not in all_class_dirs]
if missing:
    raise ValueError(f"Could not find these classes in the dataset: {missing}. "
                      f"Available: {sorted(all_class_dirs.keys())}")

SPLIT_DIR = DATA_ROOT / "rsh_split"
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

rng = random.Random(SEED)
counts = {}
for cls in TARGET_CLASSES:
    src_dir = all_class_dirs[cls]
    files = [f for f in src_dir.iterdir() if f.is_file()]
    rng.shuffle(files)

    n = len(files)
    n_train = int(n * SPLIT_RATIOS[0])
    n_val = int(n * SPLIT_RATIOS[1])
    splits = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    counts[cls] = {k: len(v) for k, v in splits.items()}

    for split_name, split_files in splits.items():
        out_dir = SPLIT_DIR / split_name / cls
        out_dir.mkdir(parents=True, exist_ok=True)
        for f in split_files:
            shutil.copy(f, out_dir / f.name)

print(f"{'class':8s} {'train':>7s} {'val':>7s} {'test':>7s}")
for cls, d in counts.items():
    print(f"{cls:8s} {d['train']:7d} {d['val']:7d} {d['test']:7d}")


In [ ]:
# 5. Transforms + Datasets + DataLoaders
IMG_SIZE = 224
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(SPLIT_DIR / "train", transform=train_transform)
val_ds = datasets.ImageFolder(SPLIT_DIR / "val", transform=eval_transform)
test_ds = datasets.ImageFolder(SPLIT_DIR / "test", transform=eval_transform)

# ImageFolder assigns class indices alphabetically -- keep this mapping, we need
# it again at inference time.
CLASS_NAMES = train_ds.classes
print("Class -> index mapping:", train_ds.class_to_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")


In [ ]:
# 6. Sanity-check: visualize one batch
def denormalize(img_tensor):
    img = img_tensor.numpy().transpose((1, 2, 0))
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, lbl in zip(axes.flat, images, labels):
    ax.imshow(denormalize(img))
    ax.set_title(CLASS_NAMES[lbl])
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 7. Build the model: ResNet18 pretrained on ImageNet, new classifier head
def build_model(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False  # freeze backbone for phase 1

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model.to(DEVICE)

model = build_model(num_classes=len(CLASS_NAMES))
print(model.fc)


In [ ]:
# 8. Train / evaluate helper functions
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            if is_train:
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)
            total_loss += loss.item() * inputs.size(0)
            total_correct += (preds == targets).sum().item()
            total_samples += inputs.size(0)

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, train_loader, val_loader, epochs, lr, weight_decay=1e-4):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"epoch {epoch:2d}/{epochs}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}  "
              f"({time.time()-t0:.1f}s)")

    model.load_state_dict(best_state)
    return model, history


In [ ]:
# 9. Phase 1: train only the new classifier head (backbone frozen)
model, history_phase1 = train_model(
    model, train_loader, val_loader, epochs=6, lr=1e-3
)


In [ ]:
# 10. Phase 2: unfreeze everything and fine-tune with a small learning rate
for param in model.parameters():
    param.requires_grad = True

model, history_phase2 = train_model(
    model, train_loader, val_loader, epochs=10, lr=1e-4
)

# merge histories for plotting
history = {k: history_phase1[k] + history_phase2[k] for k in history_phase1}


In [ ]:
# 11. Plot training curves
# Figures go into results/ so they can be committed alongside the code
# (the README of the repo references these two paths).
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"], label="val")
axes[0].axvline(len(history_phase1["train_loss"]) + 0.5, color="gray", linestyle="--", label="unfreeze")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="train")
axes[1].plot(epochs_range, history["val_acc"], label="val")
axes[1].axvline(len(history_phase1["train_acc"]) + 0.5, color="gray", linestyle="--", label="unfreeze")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
plt.show()


In [ ]:
# 12. Final evaluation on the held-out TEST set
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.numpy())

test_acc = accuracy_score(all_targets, all_preds)
print(f"Test accuracy: {test_acc:.3%}\n")
print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion matrix (test accuracy = {test_acc:.1%})")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# 13. Look at some individual predictions (correct and wrong)
def imshow_prediction(ax, img_tensor, true_idx, pred_idx, confidence):
    ax.imshow(denormalize(img_tensor.cpu()))
    color = "green" if true_idx == pred_idx else "red"
    ax.set_title(f"true: {CLASS_NAMES[true_idx]}\npred: {CLASS_NAMES[pred_idx]} ({confidence:.0%})", color=color, fontsize=9)
    ax.axis("off")

inputs, targets = next(iter(test_loader))
inputs_dev = inputs.to(DEVICE)
with torch.no_grad():
    probs = torch.softmax(model(inputs_dev), dim=1)
    confs, preds = probs.max(dim=1)

n_show = min(8, len(inputs))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, t, p, c in zip(axes.flat, inputs[:n_show], targets[:n_show], preds[:n_show].cpu(), confs[:n_show].cpu()):
    imshow_prediction(ax, img, t.item(), p.item(), c.item())
plt.tight_layout()
plt.show()


In [ ]:
# 14. Save the trained model
MODEL_PATH = Path("rain_snow_hail_classifier.pth")
SCRIPTED_PATH = Path("rain_snow_hail_classifier_scripted.pt")

checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,          # index -> class name, in ImageFolder order
    "img_size": IMG_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "architecture": "resnet18",
    "test_accuracy": test_acc,
}
torch.save(checkpoint, MODEL_PATH)
print(f"Saved checkpoint to {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

# Also export a TorchScript version -- easiest thing to hand to someone else /
# load without needing this notebook's Python code.
model_cpu = model.to("cpu").eval()
example_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
scripted = torch.jit.trace(model_cpu, example_input)
scripted.save(str(SCRIPTED_PATH))
model.to(DEVICE)
print(f"Saved TorchScript model to {SCRIPTED_PATH}")

with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)


In [ ]:
# 15. Download the saved files (Colab only)
if IN_COLAB:
    from google.colab import files
    for fname in [MODEL_PATH, SCRIPTED_PATH,
                  RESULTS_DIR / "training_curves.png", RESULTS_DIR / "confusion_matrix.png"]:
        files.download(str(fname))
else:
    print(f"Files saved locally: {MODEL_PATH}, {SCRIPTED_PATH}, and figures in {RESULTS_DIR}/")


## 16. Live demo: classify your own photo
Run the cell below, upload any rain/snow/hail photo, and see the model's prediction with a confidence score.
This is the easiest cell to run live when showing the model to someone else.

In [ ]:
def load_classifier(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    clf = build_model(num_classes=len(ckpt["class_names"]))
    clf.load_state_dict(ckpt["model_state_dict"])
    clf.eval()
    infer_transform = transforms.Compose([
        transforms.Resize(int(ckpt["img_size"] * 1.14)),
        transforms.CenterCrop(ckpt["img_size"]),
        transforms.ToTensor(),
        transforms.Normalize(ckpt["imagenet_mean"], ckpt["imagenet_std"]),
    ])
    return clf, infer_transform, ckpt["class_names"]


def predict_image(clf, infer_transform, class_names, pil_image):
    x = infer_transform(pil_image.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(clf(x), dim=1)[0]
    order = probs.argsort(descending=True)
    return [(class_names[i], probs[i].item()) for i in order]


clf, infer_transform, class_names = load_classifier(MODEL_PATH)

if IN_COLAB:
    from google.colab import files
    print("Upload a photo to classify (rain / snow / hail):")
    uploaded = files.upload()
    for fname in uploaded:
        img = Image.open(fname)
        results = predict_image(clf, infer_transform, class_names, img)
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        title = "\n".join(f"{name}: {p:.1%}" for name, p in results)
        plt.title(title)
        plt.show()
else:
    print("Call predict_image(clf, infer_transform, class_names, PIL.Image.open('your_photo.jpg'))")


## 17. Summary & next steps

- **Model:** ResNet18 (ImageNet-pretrained), fine-tuned in two phases on 3 classes (rain, snow, hail).
- **Data:** Kaggle "Weather Image Recognition" dataset (11-class, WEAPD-derived), filtered to the 3 target classes, 70/15/15 train/val/test split.
- **Outputs saved:** `rain_snow_hail_classifier.pth` (full checkpoint), `rain_snow_hail_classifier_scripted.pt` (TorchScript, portable), `results/training_curves.png`, `results/confusion_matrix.png`.

**Ideas to push this further:**
1. **More classes:** the same dataset has 11 phenomena (dew, frost, glaze, rime, fog/smog, lightning, rainbow, sandstorm too) — extend `TARGET_CLASSES` for a broader weather-tagging model.
2. **Domain shift:** these are general internet photos. If the target deployment is a fixed camera or a specific environment, collect and label a small validation set from that source and verify accuracy holds — fine-tune on it if it doesn't.
3. **Stronger backbones:** swap `resnet18` for `efficientnet_b0` or a small ViT and compare; the rest of the pipeline is unchanged.
4. **Deployment:** ResNet18 runs comfortably on CPU in real time, so the TorchScript export can be dropped straight into an edge or server-side inference service.
